# Virtual AI Patient — тестирование диалога

Ноутбук прогоняет набор тестовых вопросов через несколько LLM-моделей
и показывает ответы рядом для ручной оценки.

**Что проверяем:**
- **Тип A — Grounding:** модель не выходит за рамки кейса, не придумывает симптомы
- **Тип B — Прогрессивное раскрытие:** на общий вопрос — минимум, на целевой — детали
- **Тип C — Защита диагноза:** модель не намекает и не называет диагноз
- **Тип D — Тон/персона:** ответы соответствуют заданному тону и персоне

## 0. Установка зависимостей

In [ ]:
%pip install openai anthropic python-dotenv pandas ipywidgets -q

## 1. Конфигурация — ключи и модели

In [ ]:
import os

# Вставьте ключи здесь или создайте файл .env рядом с ноутбуком:
# OPENAI_API_KEY=sk-...
# ANTHROPIC_API_KEY=sk-ant-...

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY", "YOUR_OPENAI_KEY_HERE")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "YOUR_ANTHROPIC_KEY_HERE")

# Модели для сравнения
MODELS = [
    {"provider": "openai",    "model": "gpt-4o",           "label": "GPT-4o"},
    {"provider": "openai",    "model": "gpt-4o-mini",      "label": "GPT-4o-mini"},
    {"provider": "anthropic", "model": "claude-sonnet-4-20250514", "label": "Claude Sonnet"},
]

print("Конфигурация загружена.")
print(f"Модели: {[m['label'] for m in MODELS]}")

## 2. Системный промпт пациента

In [ ]:
import json
from pathlib import Path

_config_path = Path.cwd() / "case_config.json"
with open(_config_path, encoding="utf-8") as _cf:
    _case = json.load(_cf)
SYSTEM_PROMPT = _case["system_prompt"]

print("Системный промпт загружен из case_config.json.")
print(f"Длина: {len(SYSTEM_PROMPT)} символов")

## 3. Тестовые вопросы

Четыре типа:
- **A** — Grounding (проверка что не выходит за рамки кейса)
- **B** — Прогрессивное раскрытие
- **C** — Защита диагноза
- **D** — Тон и персона

In [ ]:
TEST_QUESTIONS = [
    # --- Тип A: Grounding — симптом которого НЕТ в кейсе ---
    {
        "id": "A1",
        "type": "A — Grounding",
        "question": "У вас была тошнота?",
        "expected": "Должен отрицать (тошноты нет в кейсе)",
    },
    {
        "id": "A2",
        "type": "A — Grounding",
        "question": "Температура поднималась?",
        "expected": "Должен сказать не знаю / не замечал (не задано в кейсе)",
    },
    {
        "id": "A3",
        "type": "A — Grounding",
        "question": "Вы принимаете какие-нибудь лекарства?",
        "expected": "Должен сказать нет (задано явно)",
    },

    # --- Тип B: Прогрессивное раскрытие ---
    {
        "id": "B1",
        "type": "B — Раскрытие (широкий вопрос)",
        "question": "Что вас беспокоит?",
        "expected": "Должен дать минимум: боль в груди, испуган. Без иррадиации и деталей.",
    },
    {
        "id": "B2",
        "type": "B — Раскрытие (целевой вопрос)",
        "question": "Куда отдаёт боль?",
        "expected": "Должен сказать про обе руки (конкретика по прямому вопросу)",
    },
    {
        "id": "B3",
        "type": "B — Раскрытие (целевой вопрос)",
        "question": "Что усиливает или облегчает боль?",
        "expected": "Должен сказать: хуже лёжа/при ходьбе, лучше сидя/наклонясь вперёд",
    },

    # --- Тип C: Защита диагноза ---
    {
        "id": "C1",
        "type": "C — Защита диагноза",
        "question": "Как вы думаете, что с вами происходит?",
        "expected": "Не должен называть диагноз. Должен выразить страх и незнание.",
    },
    {
        "id": "C2",
        "type": "C — Защита диагноза",
        "question": "Врач в скорой говорил что-то про сердце?",
        "expected": "Не задано в кейсе — должен ответить не знаю / не помню",
    },

    # --- Тип D: Тон и персона ---
    {
        "id": "D1",
        "type": "D — Тон/персона",
        "question": "Расскажите немного о себе — кем работаете, как живёте?",
        "expected": "Студент, 24 года, тревожный тон. Кратко, не лекция.",
    },
    {
        "id": "D2",
        "type": "D — Тон/персона",
        "question": "Не волнуйтесь, всё будет хорошо. Расскажите подробнее.",
        "expected": "Тревожный пациент. Не должен резко успокоиться и выдать всё сразу.",
    },
]

print(f"Загружено вопросов: {len(TEST_QUESTIONS)}")
for q in TEST_QUESTIONS:
    print(f"  [{q['id']}] {q['type']}: {q['question']}")

## 4. Функции вызова моделей

In [ ]:
import openai
import anthropic

openai_client    = openai.OpenAI(api_key=OPENAI_API_KEY)
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)


def call_model(provider: str, model: str, system: str, user_message: str) -> str:
    """Вызов модели — возвращает текст ответа."""
    try:
        if provider == "openai":
            response = openai_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system",  "content": system},
                    {"role": "user",    "content": user_message},
                ],
                temperature=0.7,
                max_tokens=300,
            )
            return response.choices[0].message.content.strip()

        elif provider == "anthropic":
            response = anthropic_client.messages.create(
                model=model,
                system=system,
                messages=[{"role": "user", "content": user_message}],
                temperature=0.7,
                max_tokens=300,
            )
            return response.content[0].text.strip()

        else:
            return f"[Неизвестный провайдер: {provider}]"

    except Exception as e:
        return f"[ОШИБКА: {e}]"


print("Функции вызова моделей готовы.")

## 5. Запуск тестов

In [ ]:
import pandas as pd
from IPython.display import display

results = []  # список строк для таблицы

total = len(TEST_QUESTIONS) * len(MODELS)
done  = 0

for q in TEST_QUESTIONS:
    row = {
        "id":       q["id"],
        "type":     q["type"],
        "question": q["question"],
        "expected": q["expected"],
    }
    for m in MODELS:
        print(f"[{q['id']}] {m['label']} ...", end=" ", flush=True)
        answer = call_model(
            provider=m["provider"],
            model=m["model"],
            system=SYSTEM_PROMPT,
            user_message=q["question"],
        )
        row[m["label"]] = answer
        done += 1
        print(f"готово ({done}/{total})")

    results.append(row)

df = pd.DataFrame(results)
print("\n✅ Все вопросы прогнаны.")

## 6. Просмотр результатов

In [ ]:
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.max_rows", 100)

model_cols = [m["label"] for m in MODELS]
display(df[["id", "type", "question", "expected"] + model_cols])

## 7. Постатейный просмотр — удобно для чтения

In [ ]:
DIVIDER = "─" * 70

for _, row in df.iterrows():
    print(DIVIDER)
    print(f"[{row['id']}] {row['type']}")
    print(f"Вопрос:   {row['question']}")
    print(f"Ожидание: {row['expected']}")
    print()
    for col in model_cols:
        print(f"  {col}:")
        print(f"  {row[col]}")
        print()

print(DIVIDER)

## 8. Ручная оценка

Для каждого вопроса и каждой модели поставь оценку:
- **2** — отлично, полностью соответствует ожидаемому поведению
- **1** — частично, есть мелкие проблемы
- **0** — плохо, нарушает правила кейса

In [ ]:
# Заполни вручную после просмотра ответов выше.
# Формат: { "id_вопроса": { "Название модели": оценка_0_1_2 } }

SCORES = {
    "A1": {"GPT-4o": None, "GPT-4o-mini": None, "Claude Sonnet": None},
    "A2": {"GPT-4o": None, "GPT-4o-mini": None, "Claude Sonnet": None},
    "A3": {"GPT-4o": None, "GPT-4o-mini": None, "Claude Sonnet": None},
    "B1": {"GPT-4o": None, "GPT-4o-mini": None, "Claude Sonnet": None},
    "B2": {"GPT-4o": None, "GPT-4o-mini": None, "Claude Sonnet": None},
    "B3": {"GPT-4o": None, "GPT-4o-mini": None, "Claude Sonnet": None},
    "C1": {"GPT-4o": None, "GPT-4o-mini": None, "Claude Sonnet": None},
    "C2": {"GPT-4o": None, "GPT-4o-mini": None, "Claude Sonnet": None},
    "D1": {"GPT-4o": None, "GPT-4o-mini": None, "Claude Sonnet": None},
    "D2": {"GPT-4o": None, "GPT-4o-mini": None, "Claude Sonnet": None},
}

print("Заполни SCORES выше и запусти следующую ячейку для итогов.")

## 9. Итоговая таблица оценок

In [ ]:
score_rows = []
for qid, model_scores in SCORES.items():
    q_info = next(q for q in TEST_QUESTIONS if q["id"] == qid)
    row = {"id": qid, "type": q_info["type"], "question": q_info["question"]}
    row.update(model_scores)
    score_rows.append(row)

df_scores = pd.DataFrame(score_rows)
display(df_scores)

print("\n── Суммарные баллы по моделям ──")
max_score = len(TEST_QUESTIONS) * 2
for col in model_cols:
    total_score = df_scores[col].sum()
    filled = df_scores[col].notna().sum()
    print(f"  {col}: {total_score} / {filled * 2}  (заполнено {filled}/{len(TEST_QUESTIONS)})")

print("\n── Слабые места по типам вопросов ──")
for q_type in df_scores["type"].unique():
    subset = df_scores[df_scores["type"] == q_type][model_cols]
    means = subset.mean()
    print(f"  {q_type}:")
    for col, val in means.items():
        print(f"    {col}: {val:.1f}" if val is not None else f"    {col}: —")

## 10. Интерактивный диалог с выбранной моделью

Свободный чат для углублённого тестирования конкретной модели.

In [ ]:
# Выбери модель для диалога
CHAT_MODEL = MODELS[0]  # поменяй индекс: 0=GPT-4o, 1=GPT-4o-mini, 2=Claude

chat_history = []  # история диалога (только user/assistant, без system)

def chat(user_message: str) -> str:
    """Отправить сообщение и получить ответ с сохранением истории."""
    chat_history.append({"role": "user", "content": user_message})

    try:
        if CHAT_MODEL["provider"] == "openai":
            messages = [{"role": "system", "content": SYSTEM_PROMPT}] + chat_history
            response = openai_client.chat.completions.create(
                model=CHAT_MODEL["model"],
                messages=messages,
                temperature=0.7,
                max_tokens=300,
            )
            reply = response.choices[0].message.content.strip()

        elif CHAT_MODEL["provider"] == "anthropic":
            response = anthropic_client.messages.create(
                model=CHAT_MODEL["model"],
                system=SYSTEM_PROMPT,
                messages=chat_history,
                temperature=0.7,
                max_tokens=300,
            )
            reply = response.content[0].text.strip()

    except Exception as e:
        reply = f"[ОШИБКА: {e}]"

    chat_history.append({"role": "assistant", "content": reply})

    print(f"👨‍⚕️ Врач:    {user_message}")
    print(f"🤒 Пациент: {reply}")
    print()
    return reply


def reset_chat():
    """Сбросить историю диалога."""
    chat_history.clear()
    print("История диалога сброшена.")


print(f"Диалог с моделью: {CHAT_MODEL['label']}")
print("Используй: chat('ваш вопрос')  |  reset_chat() для сброса")

In [ ]:
# Пример диалога — раскомментируй и запускай последовательно

# chat("Здравствуйте. Что вас привело сегодня?")

In [ ]:
# chat("Куда отдаёт боль?")

In [ ]:
# chat("У вас была тошнота?")

In [ ]:
# chat("Как вы думаете, что с вами?")

In [ ]:
# reset_chat()  # сброс перед новым сценарием